# Predicting Medical Insurance Charges
### Your first end-to-end regression model

By the end of this session you will have trained a model, measured how wrong it is,
worked out *why* it was wrong, and fixed it — and you will be able to explain every
step of that in plain English.

**Session plan**

| Phase | What we do | Time |
|---|---|---|
| 1 | Explore the data | 30 min |
| 2 | Train a deliberately bad baseline | 25 min |
| 3 | Diagnose why it's bad | 20 min |
| 4 | Improve it, one change at a time | 35 min |
| 5 | Read the model out loud | 10 min |

**Dataset**: Medical Insurance Cost Prediction (2,772 rows, 7 columns, MIT licence)
`kaggle.com/datasets/rahulvyasm/medical-insurance-cost-prediction`

---
## Setup

Everything here is pre-installed in Google Colab. Run this cell and wait for the `*` to disappear.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
print('Ready.')

## Loading the data

The cell below pulls the file straight from Hugging Face so nobody needs a Kaggle login.
If your network blocks it, download `medical_insurance.csv` from the Kaggle link above,
upload it with the folder icon on the left of Colab, and the fallback will pick it up.

In [ ]:
def load_insurance_data():
    """Try Hugging Face first, fall back to a local file."""
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
        repo = 'rahulvyasm/medical_insurance_data'
        csvs = [f for f in list_repo_files(repo, repo_type='dataset') if f.endswith('.csv')]
        path = hf_hub_download(repo, csvs[0], repo_type='dataset')
        print(f'Loaded from Hugging Face: {csvs[0]}')
        return pd.read_csv(path)
    except Exception as e:
        print(f'Hugging Face unavailable ({type(e).__name__}), trying local file...')
        return pd.read_csv('medical_insurance.csv')

df = load_insurance_data()
df.head()

---
# Phase 1 — Explore the data (30 min)

**Rule for this phase: no modelling.** We are only trying to understand what we have.
A model built on data you haven't looked at is a guess with extra steps.

In [ ]:
print('Rows, columns:', df.shape)
print()
df.info()

In [ ]:
df.describe()

### The first real check

Before anything else: are there missing values, and are there **duplicate rows**?

Duplicates matter more than people expect. If the same row appears twice and the
train/test split sends one copy to training and the other to testing, the model gets
to memorise an answer and then take a test containing that exact answer. The score
goes up and means nothing. This is a mild form of *data leakage*.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())

> **Instructor note.** This dataset is an expanded re-upload of the well-known 1,338-row
> `insurance.csv`. Expect duplicates. Ask the room what they think should happen before
> you run the next cell — it is one of the few moments where the *right* answer is
> genuinely arguable. Two different people really could be the same age, sex, BMI, region,
> smoking status **and** be billed an identical amount to five decimal places... but it's
> very unlikely. We drop them.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(df):,} duplicate rows. {len(df):,} remain.')

### What does the thing we're predicting look like?

`charges` is our **target**. Everything else is a candidate **feature**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['charges'], bins=50, ax=axes[0])
axes[0].set_title('Distribution of charges')
axes[0].set_xlabel('charges ($)')
sns.boxplot(x=df['charges'], ax=axes[1])
axes[1].set_title('Charges — spread and outliers')
axes[1].set_xlabel('charges ($)')
plt.tight_layout(); plt.show()

print(df['charges'].describe())

**Notice the shape.** Most people cost a modest amount, and a long tail costs a great
deal. This is a *right-skewed* distribution and it is extremely common in anything
involving money. Park that observation — we come back to it in Phase 4.

### The relationship we came here for

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='age', y='charges', alpha=0.4)
plt.title('Age vs charges')
plt.show()

**Stop and look at this properly.**

There is an upward trend — older people cost more. But this is not one cloud of points
around one line. There appear to be *bands*: a dense lower band, and one or two
sparser bands sitting well above it.

That structure is the whole session. Something is splitting these people into groups,
and `age` alone cannot see it.

**Ask the room: what could it be?** Take guesses before running the next cell.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='age', y='charges', hue='smoker', alpha=0.5)
plt.title('Age vs charges, coloured by smoking status')
plt.show()

There it is. Now let's put a number on it instead of relying on the picture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x='smoker', y='charges', ax=axes[0])
axes[0].set_title('Charges by smoking status')
sns.scatterplot(data=df, x='bmi', y='charges', hue='smoker', alpha=0.5, ax=axes[1])
axes[1].set_title('BMI vs charges, coloured by smoking status')
plt.tight_layout(); plt.show()

print(df.groupby('smoker')['charges'].agg(['count', 'mean', 'median']))

> **Instructor note — the BMI plot on the right is the subtle one.**
> For non-smokers, BMI barely moves charges: the points stay flat as you go right.
> For smokers, charges climb steeply with BMI. The effect of BMI *depends on whether
> you smoke*. That is called an **interaction**, and plain linear regression cannot
> represent it unless we explicitly build it. We do that in Phase 4, Step 3.

### Do the other columns matter?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['sex', 'region', 'children']):
    sns.boxplot(data=df, x=col, y='charges', ax=ax)
    ax.set_title(f'Charges by {col}')
plt.tight_layout(); plt.show()

In [ ]:
df[['age', 'bmi', 'children', 'charges']].corr()['charges'].sort_values(ascending=False)

### Checkpoint — answer these before moving on

1. Which single column looks like the strongest driver of charges?
2. Which columns look almost irrelevant?
3. Why is the correlation between `age` and `charges` so weak, when the scatter plot
   clearly slopes upward?

*(Answer to 3: correlation measures how well a **single straight line** fits. Three
separate bands are badly described by one line, so the number comes out low even though
a real relationship exists. The statistic isn't lying — it's answering a narrower
question than the one we care about.)*

---
# Phase 2 — A deliberately bad baseline (25 min)

We are going to start with the worst defensible model: predict `charges` from `age`
and nothing else.

This is not wasted time. A baseline is the number every later improvement gets measured
against. Without one, "my model got 0.87" is a statement with no meaning.

### Our scorecard

Three metrics, and you should be able to say what each one means:

- **R²** — what fraction of the variation in charges the model explains. 1.0 is perfect, 0.0 is no better than always guessing the average.
- **MAE** — mean absolute error. The average size of our mistake, in dollars. Easiest to explain to a non-technical person.
- **RMSE** — root mean squared error. Also in dollars, but it punishes large mistakes much harder than small ones.

In [ ]:
scorecard = []

def record(name, y_true, y_pred):
    """Score a model, add it to the running scorecard, and show the table."""
    scorecard.append({
        'model': name,
        'R2': r2_score(y_true, y_pred),
        'MAE ($)': mean_absolute_error(y_true, y_pred),
        'RMSE ($)': np.sqrt(mean_squared_error(y_true, y_pred)),
    })
    return pd.DataFrame(scorecard).set_index('model')

print('Scorecard ready.')

### Splitting the data

We hold back 20% of the rows and never let the model see them during training. Testing
a model on data it trained on is like marking your own exam with the answer sheet open.

`random_state=42` just makes the split reproducible — everyone in the room gets the same
rows, so everyone gets the same numbers.

In [ ]:
X_base = df[['age']]
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42)

print(f'Training on {len(X_train):,} rows, testing on {len(X_test):,} rows.')

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)
record('1. age only', y_test, y_pred)

### What the model actually learned

A linear regression with one feature is just the equation of a straight line:

$$\text{charges} = \text{intercept} + \text{slope} \times \text{age}$$

In [ ]:
print(f'Intercept: ${baseline.intercept_:,.2f}')
print(f'Slope:     ${baseline.coef_[0]:,.2f} per year of age')
print()
print(f'So the model says: a 40-year-old costs about '
      f'${baseline.intercept_ + baseline.coef_[0]*40:,.2f}')

The slope is a real, sensible finding — each additional year of age is associated with
a few hundred dollars more in charges. But look back at the R² on the scorecard. We are
explaining almost none of what's going on.

**A model can be right about something and still be useless.**

---
# Phase 3 — Why is it bad? (20 min)

Do not skip to fixing it. Learning to interrogate a bad model is the skill; the fix is
just typing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4)
lims = [0, max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', label='perfect prediction')
axes[0].set_xlabel('Actual charges ($)')
axes[0].set_ylabel('Predicted charges ($)')
axes[0].set_title('Predicted vs actual')
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted charges ($)')
axes[1].set_ylabel('Residual (actual - predicted, $)')
axes[1].set_title('Residual plot')

plt.tight_layout(); plt.show()

### How to read these two plots

**Left.** Every point should sit on the red line. Instead they form a vertical smear —
for almost any prediction the model makes, the true answer could be almost anything.

**Right.** This is the one that matters. A residual is how much we missed by. If a model
has captured everything there is to capture, residuals should look like **random,
structureless noise** scattered evenly around zero.

Ours don't. There is visible banding — the same bands we saw in the very first scatter
plot. **Structure left in the residuals is signal the model hasn't learned yet.**

That is the diagnosis. Now let's treat it.

In [ ]:
# Just how skewed are our errors? Notice they aren't centred symmetrically.
print(f'Mean residual:   ${residuals.mean():>12,.2f}')
print(f'Median residual: ${residuals.median():>12,.2f}')
print(f'Worst under-prediction: ${residuals.max():>12,.2f}')
print(f'Worst over-prediction:  ${residuals.min():>12,.2f}')

---
# Phase 4 — Improve it (35 min)

**One change at a time, re-score after each.** If you change three things at once and
the score moves, you have learned nothing about which change helped.

## Step 2 — Use all the features

Our model can't use `sex`, `smoker` or `region` yet, because those are text and the
maths needs numbers. **One-hot encoding** turns each category into a 0/1 column.

`drop='first'` removes one column per category to avoid redundancy — if `smoker_yes` is
0, we already know the person is a non-smoker, so a `smoker_no` column adds nothing.

We wrap this in a **Pipeline** so the encoding is learned from training data only and
applied identically to test data. This is not decoration — doing it by hand is one of
the most common sources of leakage in real projects.

In [ ]:
numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features),
])

X_full = df[numeric_features + categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42)

model_2 = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression()),
])
model_2.fit(X_train, y_train)

record('2. all features', y_test, model_2.predict(X_test))

A large jump. Almost all of it comes from one column — `smoker` — which is exactly what
the EDA predicted. **This is why we spent 30 minutes looking at plots.**

## Step 3 — Add the interaction we spotted

Remember the BMI plot: BMI barely mattered for non-smokers but mattered a lot for
smokers. A plain linear model assumes every feature has the same effect on everyone.
We have to build the exception by hand, by multiplying the two columns together.

In [ ]:
df_fe = df.copy()
df_fe['smoker_bin'] = (df_fe['smoker'] == 'yes').astype(int)
df_fe['bmi_x_smoker'] = df_fe['bmi'] * df_fe['smoker_bin']

numeric_fe = ['age', 'bmi', 'children', 'bmi_x_smoker']

preprocessor_fe = ColumnTransformer([
    ('num', 'passthrough', numeric_fe),
    ('cat', OneHotEncoder(drop='first'), categorical_features),
])

X_fe = df_fe[numeric_fe + categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, random_state=42)

model_3 = Pipeline([
    ('prep', preprocessor_fe),
    ('model', LinearRegression()),
])
model_3.fit(X_train, y_train)

record('3. + BMI x smoker', y_test, model_3.predict(X_test))

Another jump — from a feature we invented ourselves by looking at a picture. No new
data was collected. This is **feature engineering**, and on structured business data it
beats swapping in a fancier algorithm more often than beginners expect.

## Step 4 — Scaling: what it really does

Scaling (standardising every feature to mean 0, standard deviation 1) gets recommended
constantly. So let's test the recommendation instead of repeating it.

**Prediction before you run the cell:** does adding a `StandardScaler` improve our R²?

In [ ]:
model_raw = Pipeline([
    ('prep', preprocessor_fe),
    ('model', LinearRegression()),
]).fit(X_train, y_train)

model_scaled = Pipeline([
    ('prep', preprocessor_fe),
    ('scale', StandardScaler()),
    ('model', LinearRegression()),
]).fit(X_train, y_train)

print(f'R2 without scaling: {r2_score(y_test, model_raw.predict(X_test)):.12f}')
print(f'R2 with scaling:    {r2_score(y_test, model_scaled.predict(X_test)):.12f}')
print()
diff = np.abs(model_raw.predict(X_test) - model_scaled.predict(X_test)).max()
print(f'Largest difference between any two predictions: ${diff:.10f}')

### Identical. To twelve decimal places.

**Scaling does not improve plain linear regression, and it never will.** Ordinary least
squares has an exact mathematical solution; stretching the axes changes the coefficients
but rescales them right back to the same predictions.

If you have seen a tutorial that scales the data and reports a better score, the
improvement came from something else in the same cell.

**So why does everyone tell you to scale?** Two real reasons:

1. **Comparable coefficients.** Unscaled, a coefficient of 250 on `age` and 300 on `bmi` tells you nothing about which matters more, because a year and a BMI point aren't comparable units. Scaled, they are.
2. **Some models genuinely require it** — anything with a penalty term or gradient descent. Which brings us to the next cell.

## Step 5 — Where scaling actually bites: Ridge

**Ridge regression** is linear regression with a penalty on large coefficients, to stop
the model leaning too hard on any one feature. That penalty is applied to the raw
coefficient values — so a feature measured in large units gets penalised differently
from one measured in small units. Here, scaling changes the answer.

In [ ]:
ridge_raw = Pipeline([
    ('prep', preprocessor_fe),
    ('model', Ridge(alpha=100)),
]).fit(X_train, y_train)

ridge_scaled = Pipeline([
    ('prep', preprocessor_fe),
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=100)),
]).fit(X_train, y_train)

print(f'Ridge R2 without scaling: {r2_score(y_test, ridge_raw.predict(X_test)):.6f}')
print(f'Ridge R2 with scaling:    {r2_score(y_test, ridge_scaled.predict(X_test)):.6f}')

record('4. Ridge (scaled)', y_test, ridge_scaled.predict(X_test))

**Different numbers.** Same data, same algorithm, same penalty — only the scaling
changed. That is the honest lesson about preprocessing: it isn't a magic score booster
you sprinkle on everything, it's a step that some specific models require in order to
behave sensibly.

> Note that Ridge probably didn't beat plain linear regression here. With 2,700 rows and
> eight features we aren't overfitting, so there's nothing for the penalty to fix.
> Regularisation earns its place when features are many and rows are few.

## Step 6 — Optional experiment: the skewed target

Back in Phase 1 we noticed `charges` has a long right tail. Linear regression is
happiest when errors are evenly spread, and a skewed target works against that. A
standard response is to model `log(charges)` instead, then convert back to dollars.

**This may or may not help here — run it and see.** Reporting an experiment that didn't
work is a completely respectable result, and pretending otherwise is how bad habits start.

In [ ]:
y_log = np.log(y)

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_fe, y_log, test_size=0.2, random_state=42)

model_log = Pipeline([
    ('prep', preprocessor_fe),
    ('model', LinearRegression()),
]).fit(X_tr_l, y_tr_l)

# Convert predictions back from log-space into dollars before scoring,
# otherwise we'd be comparing against a different scale entirely.
pred_dollars = np.exp(model_log.predict(X_te_l))

record('5. log(charges) target', y_test, pred_dollars)

---
# Phase 5 — Read the model out loud (10 min)

A model nobody can explain doesn't get used. This is the part that makes you useful in
a room full of non-technical people.

In [ ]:
feature_names = model_3.named_steps['prep'].get_feature_names_out()
coefs = pd.DataFrame({
    'feature': [n.split('__')[-1] for n in feature_names],
    'coefficient': model_3.named_steps['model'].coef_,
}).sort_values('coefficient', key=abs, ascending=False).reset_index(drop=True)

coefs

### How to read this table — including one trap

For most rows the reading is simple: **holding everything else constant, a one-unit
increase in this feature changes predicted charges by this many dollars.** One more year
of age, a few hundred dollars. One more child, a few hundred dollars.

**The trap.** Once you add an interaction term, you can no longer read `smoker_yes` on
its own as "the cost of smoking". The effect of smoking is now split across two rows:

$$\text{smoking effect} = \text{coef}(\texttt{smoker\_yes}) + \text{coef}(\texttt{bmi\_x\_smoker}) \times \text{BMI}$$

You'll likely see a large *negative* `smoker_yes` coefficient paired with a positive
interaction coefficient, which looks alarming until you realise they only make sense
added together at a real BMI value. The cell below does that arithmetic.

In [ ]:
c = dict(zip([n.split('__')[-1] for n in feature_names],
             model_3.named_steps['model'].coef_))

for bmi_value in [22, 28, 35, 42]:
    effect = c['smoker_yes'] + c['bmi_x_smoker'] * bmi_value
    print(f'At BMI {bmi_value}: smoking is associated with ${effect:>12,.0f} in extra charges')

**That is your headline finding**, and it's a sentence anyone can understand:
smoking costs you, and it costs you far more the higher your BMI is.

One word of care before anyone repeats it outside this room: this is an **association**
in one dataset, not a proven cause, and not a claim about any individual person.

### Predicting for one new person

In [ ]:
new_person = pd.DataFrame([{
    'age': 45,
    'bmi': 31.0,
    'children': 2,
    'sex': 'male',
    'smoker': 'no',
    'region': 'southeast',
}])
new_person['bmi_x_smoker'] = new_person['bmi'] * (new_person['smoker'] == 'yes')
new_person = new_person[numeric_fe + categorical_features]

prediction = model_3.predict(new_person)[0]
print(f'Predicted annual charges: ${prediction:,.2f}')

# Same person, but a smoker
smoker_version = new_person.copy()
smoker_version['smoker'] = 'yes'
smoker_version['bmi_x_smoker'] = smoker_version['bmi']
print(f'Same person as a smoker:  ${model_3.predict(smoker_version)[0]:,.2f}')

### The journey, in one table

In [ ]:
pd.DataFrame(scorecard).set_index('model')

### Saving the model

The whole pipeline — encoding and model together — saves as one object. Feed it raw
data in the original format and it handles everything. This is the file we'll deploy
alongside the classification model later in the course.

In [ ]:
import joblib

joblib.dump(model_3, 'insurance_charges_model.joblib')
print('Saved to insurance_charges_model.joblib')

# Prove it survives the round trip
reloaded = joblib.load('insurance_charges_model.joblib')
print(f'Reloaded model predicts: ${reloaded.predict(new_person)[0]:,.2f}')

---
## Your turn

**1.** Re-run the whole notebook with `random_state=1` instead of `42`. How much do the
scores move? What does that tell you about trusting a single number to three decimals?

**2.** We dropped duplicate rows early on. Skip that step and re-run. Does the test score
go up or down, and — more importantly — is the model actually better?

**3.** Add an `age²` feature (`df_fe['age_sq'] = df_fe['age'] ** 2`). Charges may rise
faster in later years than earlier ones. Does it help?

**4.** Drop `region` entirely. The EDA suggested it barely matters. Does removing it hurt?

**5.** Plot the residuals of `model_3` the way we did in Phase 3. Is there structure left?
What would you try next?

---

### What you can now say you've done

Trained a regression model, measured it three ways, diagnosed its failure from a
residual plot, engineered a feature that fixed it, tested a piece of received wisdom
and found it wrong, and explained the result in a sentence a non-technical person
would understand.

That last one is the job.